In [1]:
# Configuración de Arquitectura Multi-Cabezal
import os
import sys
import multiprocessing

CACHE_DIR = "D:/Datasets_Cache"
MODELS_DIR = "../models"
OUTPUT_FILE = os.path.join(MODELS_DIR, "s2_class_map_multi.pkl") # L4 (País), L7 (Región), L10 (Ciudad)
S2_LEVELS = [4, 7, 10]                                           
MIN_IMAGES_PER_CLASS = 50                                        # Mínimo de fotos para aceptar una celda en CUALQUIER nivel
NUM_CORES = max(1, multiprocessing.cpu_count() - 1)

os.environ["HF_DATASETS_CACHE"] = CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("Verificando librerías")
os.system('pip install -q "datasets<3.0.0" s2sphere folium tqdm')
print(f"Configuración lista. Usando caché en: {CACHE_DIR}")
print(f"Niveles S2 a calcular: {S2_LEVELS}")

Verificando librerías
Configuración lista. Usando caché en: D:/Datasets_Cache
Niveles S2 a calcular: [4, 7, 10]


In [2]:
def batch_process_s2_multi(batch):
    import s2sphere
    
    tokens_l4, tokens_l7, tokens_l10 = [], [], []
    lats = batch['latitude']
    lons = batch['longitude']
    
    for lat, lon in zip(lats, lons):
        t4, t7, t10 = None, None, None
        try:
            if lat is not None and lon is not None:
                if not (abs(lat) > 90 or abs(lon) > 180 or (lat == 0 and lon == 0)):
                    p1 = s2sphere.LatLng.from_degrees(lat, lon)
                    cell = s2sphere.CellId.from_lat_lng(p1)
                    # Calculamos los 3 niveles jerárquicos
                    t4 = cell.parent(4).to_token()
                    t7 = cell.parent(7).to_token()
                    t10 = cell.parent(10).to_token()
        except Exception:
            pass 
            
        tokens_l4.append(t4)
        tokens_l7.append(t7)
        tokens_l10.append(t10)
            
    return {"token_l4": tokens_l4, "token_l7": tokens_l7, "token_l10": tokens_l10}

In [3]:
import time
import collections
import pickle
from datasets import load_dataset

def run_etl_multi():
    print("Cargando OSV-5M (Train)")
    try:
        dataset = load_dataset("osv5m/osv5m", split="train", streaming=False, trust_remote_code=True)
        cols_to_keep = {'latitude', 'longitude'}
        cols_to_drop = [c for c in dataset.column_names if c not in cols_to_keep]
        dataset = dataset.remove_columns(cols_to_drop)
    except Exception as e:
        print(f"Error cargando dataset: {e}")
        return None, None

    print(f"\nIniciando cálculo S2 Multi-Nivel con {NUM_CORES} núcleos")
    start_time = time.time()
    
    # Mapeamos los 3 niveles
    results = dataset.map(
        batch_process_s2_multi,
        batched=True,
        batch_size=5000,
        num_proc=NUM_CORES,
        remove_columns=['latitude', 'longitude'],
        desc="Generando Tokens L4, L7 y L10"
    )

    print(f"Procesado en {time.time() - start_time:.2f} segundos. Construyendo diccionarios")
    
    # Función auxiliar para crear diccionarios de clases válidas
    def build_class_map(token_list):
        valid_tokens = [t for t in token_list if t is not None]
        counter = collections.Counter(valid_tokens)
        valid_cells = [cell for cell, count in counter.items() if count >= MIN_IMAGES_PER_CLASS]
        valid_cells.sort(key=lambda x: counter[x], reverse=True)
        
        cell_to_idx = {cell: idx for idx, cell in enumerate(valid_cells)}
        return cell_to_idx, len(valid_cells)

    # Construimos los 3 mapas
    map_l4, total_l4 = build_class_map(results['token_l4'])
    map_l7, total_l7 = build_class_map(results['token_l7'])
    map_l10, total_l10 = build_class_map(results['token_l10'])
    
    final_data = {
        "L4": {"cell_to_idx": map_l4, "total_cells": total_l4},
        "L7": {"cell_to_idx": map_l7, "total_cells": total_l7},
        "L10": {"cell_to_idx": map_l10, "total_cells": total_l10}
    }
    
    with open(OUTPUT_FILE, "wb") as f:
        pickle.dump(final_data, f)
        
    print("\n--- RESUMEN DE CLASES GENERADAS ---")
    print(f"Nivel 4 (Continente/País): {total_l4:,} clases")
    print(f"Nivel 7 (Región/Estado)  : {total_l7:,} clases")
    print(f"Nivel 10 (Ciudad/Local)  : {total_l10:,} clases")
    print(f"Archivo guardado: {os.path.abspath(OUTPUT_FILE)}")
    
    return OUTPUT_FILE, results

output_pkl, ds_with_tokens = run_etl_multi()

d:\Projects\GeoGuessrAI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando OSV-5M (Train)
OSV5M {'full': False, 'name': 'osv5m', 'hash': '0ac7fc681aace0f00245c6dea7848bcb0e424845081905154c1adb5f9bf8f19e', 'base_path': 'https://huggingface.co/datasets/osv5m/osv5m/resolve/main', 'token': None, 'use_auth_token': None, 'repo_id': 'osv5m/osv5m', 'storage_options': {}, 'dataset_name': 'osv5m', '_writer_batch_size': None, 'config_kwargs': {}, 'config': BuilderConfig(name='default', version=0.0.0, data_dir=None, data_files=None, description=None), 'config_id': 'default', 'info': DatasetInfo(description='', citation='', homepage='', license='', features={'image': Image(mode=None, decode=True, id=None), 'latitude': Value(dtype='float32', id=None), 'longitude': Value(dtype='float32', id=None), 'country': Value(dtype='string', id=None), 'region': Value(dtype='string', id=None), 'sub-region': Value(dtype='string', id=None), 'city': Value(dtype='string', id=None)}, post_processed=None, supervised_keys=None, task_templates=None, builder_name='osv5m', dataset_name='

Generando Tokens L4, L7 y L10 (num_proc=7): 100%|██████████| 4893782/4893782 [00:37<00:00, 131110.70 examples/s]


Procesado en 37.53 segundos. Construyendo diccionarios

--- RESUMEN DE CLASES GENERADAS ---
Nivel 4 (Continente/País): 423 clases
Nivel 7 (Región/Estado)  : 5,705 clases
Nivel 10 (Ciudad/Local)  : 26,361 clases
Archivo guardado: d:\Projects\GeoGuessrAI\models\s2_class_map_multi.pkl


In [4]:
from tqdm import tqdm

print("Generando índices de entrenamiento")

# Solo exigimos que el Nivel 4 sea válido para guardar la imagen
with open(OUTPUT_FILE, "rb") as f:
    class_data = pickle.load(f)

valid_l4_cells = set(class_data["L4"]["cell_to_idx"].keys())
train_indices = []

for idx, token_l4 in enumerate(tqdm(ds_with_tokens['token_l4'], desc="Filtrando dataset")):
    if token_l4 in valid_l4_cells:
        train_indices.append(idx)

INDICES_FILE = os.path.join(MODELS_DIR, "train_indices_multi.pkl")
with open(INDICES_FILE, "wb") as f:
    pickle.dump(train_indices, f)

print(f"\n¡Éxito! Índices guardados en: {os.path.abspath(INDICES_FILE)}")
print(f"Imágenes recuperadas para entrenar: {len(train_indices):,} de {len(ds_with_tokens):,}")

Generando índices de entrenamiento


Filtrando dataset: 100%|██████████| 4893782/4893782 [00:01<00:00, 2509623.45it/s]



¡Éxito! Índices guardados en: d:\Projects\GeoGuessrAI\models\train_indices_multi.pkl
Imágenes recuperadas para entrenar: 4,892,351 de 4,893,782
